<a href="https://colab.research.google.com/github/GabrielJ07/ConfiguratorAgent/blob/main/Convert_MyActivity_to_Markdown.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
from datetime import datetime
import os
from html_utils import clean_html

def clean_title(title):
    """Clean up the \"Prompted \" or \"Said \" prefix Google Takeout automatically adds."""
    if title.startswith("Prompted "):
        return title[9:]
    elif title.startswith("Said "):
        return title[5:]
    return title

def format_timestamp(time_str):
    """Convert ISO time to a readable format."""
    try:
        # Convert ISO time to a readable format
        dt = datetime.fromisoformat(time_str.replace('Z', '+00:00'))
        return dt.strftime("%B %d, %Y at %I:%M %p")
    except ValueError:
        return time_str

def extract_response(item):
    """Extract and clean the AI's response from safeHtmlItem."""
    response_html = ""
    safe_html_items = item.get('safeHtmlItem', [])
    for html_item in safe_html_items:
        if 'html' in html_item:
            response_html += html_item['html'] + "\n"

    return clean_html(response_html)

def format_markdown_entry(formatted_time, title, response_text):
    """Generate the markdown string for a single entry."""
    entry = f"## Date: {formatted_time}\n"
    entry += f"**User Prompt:**\n{title}\n\n"
    if response_text:
        entry += f"**Gemini Response:**\n{response_text}\n\n"
    entry += "---\n\n"
    return entry

def clean_title(title):
    """Clean up the \"Prompted \" or \"Said \" prefix Google Takeout automatically adds."""
    if title.startswith("Prompted "):
        return title[9:]
    elif title.startswith("Said "):
        return title[5:]
    return title

def format_timestamp(time_str):
    """Convert ISO time to a readable format."""
    try:
        # Convert ISO time to a readable format
        dt = datetime.fromisoformat(time_str.replace('Z', '+00:00'))
        return dt.strftime("%B %d, %Y at %I:%M %p")
    except ValueError:
        return time_str

def extract_response(item):
    """Extract and clean the AI's response from safeHtmlItem."""
    response_html = ""
    safe_html_items = item.get('safeHtmlItem', [])
    for html_item in safe_html_items:
        if 'html' in html_item:
            response_html += html_item['html'] + "\n"

    return clean_html(response_html)

def format_markdown_entry(formatted_time, title, response_text):
    """Generate the markdown string for a single entry."""
    entry = f"## Date: {formatted_time}\n"
    entry += f"**User Prompt:**\n{title}\n\n"
    if response_text:
        entry += f"**Gemini Response:**\n{response_text}\n\n"
    entry += "---\n\n"
    return entry

def convert_myactivity_to_md(input_file, output_file):
    """Parses Google Takeout MyActivity JSON and converts it to NotebookLM-friendly Markdown."""

    if not os.path.exists(input_file):
        print(f"Error: Could not find '{input_file}'. Make sure it is in the same folder as this script.")
        return

    print(f"Reading {input_file}...")
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    markdown_lines = ["# Gemini Activity History\n\n"]
    entry_count = 0

    for item in data:
        title = item.get('title', '')
        if not title:
            continue

        title = clean_title(title)
        formatted_time = format_timestamp(item.get('time', ''))
        response_text = extract_response(item)

        # Only add entries that have actual text
        if title or response_text:
            markdown_lines.append(format_markdown_entry(formatted_time, title, response_text))
            entry_count += 1

    print(f"Writing to {output_file}...")
    with open(output_file, 'w', encoding='utf-8') as f:
        f.writelines(markdown_lines)

    print(f"Success! Converted {entry_count} conversation entries.")
    print(f"You can now upload '{output_file}' directly into NotebookLM.")

if __name__ == "__main__":
    # Ensure these file names match your local files
    INPUT_FILENAME = 'MyActivity.json'
    OUTPUT_FILENAME = 'Circuittelligence_Activity.md'

    convert_myactivity_to_md(INPUT_FILENAME, OUTPUT_FILENAME)